# RSNA Knee Abnormality Detection — Phase 4C.4 constrained synthetic canary

Internet-off Kaggle notebook. **Do not enable internet. GPU is required.**
This notebook scores **synthetic reports only**. It does **not** read the 58
gold studies or the 4,349 report-only studies. It does **not** train an image
model, create folds, or submit.

Prompt remains **v1.2.0**. Schema remains **v1.0.0**. Transport is **v1.4**
with LM Format Enforcer 0.11.3 `prefix_allowed_tokens_fn` plus the existing
JSON-object stopper and Phi EOS IDs `[32000, 32007, 32001]`.

Do not overwrite gold kernel v7, canary v1, smoke v2, or smoke/canary v1.3.
Install LM Format Enforcer only from hashed local wheels.

If the three-example smoke fails, stop. If the 14-example canary fails format
or semantic gates, stop. Do not rerun gold.


In [ ]:
# Bootstrap: stdlib only until the package is importable.
# Copies/extracts source under /kaggle/working. Never writes under /kaggle/input.
from __future__ import annotations

import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
SRC_DST = KAGGLE_WORKING / "rsna_knee_src"
OUTPUT_DIR = KAGGLE_WORKING / "rsna_second_reader_constrained_canary"
MARKERS = ("knee/config.py", "knee/second_reader_constrained.py")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"


def _looks_like_source(root: Path) -> bool:
    return all((root / marker).is_file() for marker in MARKERS)


def _safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not name or name.endswith("/"):
                continue
            target = (dest / name).resolve()
            try:
                target.relative_to(dest_resolved)
            except ValueError as exc:
                raise RuntimeError(f"refusing zip path that escapes {dest}: {name}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open("wb") as out:
                shutil.copyfileobj(src, out)


def _copy_allowlisted(src_root: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for dirname in ("knee", "scripts", "configs", "notebooks", "vendor", "third_party"):
        src = src_root / dirname
        if src.is_dir():
            shutil.copytree(src, dest / dirname, dirs_exist_ok=True)
    for filename in ("pyproject.toml", "requirements.txt", "README.md", "literature/ontology.yaml"):
        src = src_root / filename
        if src.is_file():
            dest_file = dest / filename
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest_file)


def _zip_has_source(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
    return any(any(name.endswith(marker) for marker in MARKERS) for name in names)


SKIP_WALK_DIRS = {"train_series", "test_series", "__pycache__", ".git", "competitions"}


def _walk_input_for_source(root: Path, max_depth: int = 6):
    zips: list[Path] = []
    dirs: list[Path] = []
    try:
        root_res = root.resolve()
    except OSError:
        return zips, dirs
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        try:
            rel = current.resolve().relative_to(root_res)
            depth = 0 if str(rel) == "." else len(rel.parts)
        except (OSError, ValueError):
            dirnames[:] = []
            continue
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP_WALK_DIRS and d != COMPETITION_SLUG)
        if depth > max_depth:
            dirnames[:] = []
            continue
        if _looks_like_source(current):
            dirs.append(current)
        for name in filenames:
            if name.lower().endswith(".zip"):
                zip_path = current / name
                if _zip_has_source(zip_path):
                    zips.append(zip_path)
    return zips, dirs


def locate_and_extract_source() -> Path | None:
    if _looks_like_source(SRC_DST):
        return SRC_DST
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _looks_like_source(candidate):
            if KAGGLE_WORKING.exists() and candidate != SRC_DST:
                _copy_allowlisted(candidate, SRC_DST)
                if _looks_like_source(SRC_DST):
                    return SRC_DST
            return candidate
    if not KAGGLE_INPUT.is_dir():
        return None
    zips: list[Path] = []
    dirs: list[Path] = []
    for child in sorted(KAGGLE_INPUT.iterdir()):
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            continue
        if child.is_file() and child.suffix.lower() == ".zip" and _zip_has_source(child):
            zips.append(child)
        elif child.is_dir():
            found_zips, found_dirs = _walk_input_for_source(child)
            zips.extend(found_zips)
            dirs.extend(found_dirs)
    if zips:
        print(f"extracting_source_zip={zips[0]}")
        _safe_extract(zips[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        for nested in sorted(p for p in SRC_DST.iterdir() if p.is_dir()):
            if _looks_like_source(nested):
                return nested
    if dirs:
        print(f"copying_source_tree={dirs[0]}")
        _copy_allowlisted(dirs[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        return dirs[0]
    return None


SOURCE_ROOT = locate_and_extract_source()
if SOURCE_ROOT is not None:
    resolved = str(SOURCE_ROOT.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    print(f"source_root={SOURCE_ROOT}")
else:
    print("source_root=not_found (will rely on an already importable knee package)")

try:
    from knee.kaggle_runtime import detect_kaggle_paths, KaggleLayout
    from knee.lm_format_enforcer_wheels import (
        install_from_local_wheels,
        runtime_dependency_versions,
        wheels_dir,
    )
    from knee.second_reader_constrained import (
        KAGGLE_CONSTRAINED_OUTPUT_DIRNAME,
        ensure_lmfe_transformers_compat,
    )
except ImportError as exc:
    raise SystemExit(
        "PILOT STOPPED: could not import knee. Attach the private source dataset. "
        f"Original error: {exc}"
    ) from exc

print("internet_used=False gpu_required=True training_run=False process_report_only=False synthetic_canary=True constrained=True")
print(detect_kaggle_paths(KaggleLayout(output_dirname=KAGGLE_CONSTRAINED_OUTPUT_DIRNAME)))
wheel_root = wheels_dir(SOURCE_ROOT) if SOURCE_ROOT is not None else wheels_dir()
install_payload = install_from_local_wheels(wheel_root, root=SOURCE_ROOT)
print("lmfe_install=" + json.dumps({
    k: install_payload.get(k)
    for k in ("ok", "internet_used", "version", "license", "installed", "imported_versions")
}, sort_keys=True, default=str))
print("pip_check=" + json.dumps(install_payload.get("pip_check") or {}, sort_keys=True, default=str)[:4000])
print("runtime_versions=" + json.dumps(install_payload.get("runtime_versions") or runtime_dependency_versions(), sort_keys=True, default=str))
if not install_payload.get("ok"):
    raise SystemExit("WHEEL_INSTALL_FAIL")
compat = ensure_lmfe_transformers_compat()
print("lmfe_transformers_compat=" + json.dumps(compat, sort_keys=True))


In [ ]:
# Phase 4C.4: schema audit, tokenizer prefix tree, constrained smoke, then 14x2.
# Do not read train.csv or print report text or full completions.
from pathlib import Path
import json
import time

from knee.report_labels import (
    HARD_WEAK_LABEL_TRAINING_ALLOWED,
    SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED,
)
from knee.report_second_reader import locate_weights_directory
from knee.second_reader_canary import run_synthetic_canary
from knee.second_reader_constrained import (
    aggregate_constraint_stats,
    audit_schema_against_lmfe,
    evaluate_semantic_gates,
    reset_constraint_stats,
    run_constrained_canary,
    schema_hash_provenance,
    tokenizer_prefix_tree_audit,
)
from knee.second_reader_json_stop import inspect_termination_from_snapshot
from knee.second_reader_pilot import load_pilot_config
from knee.second_reader_schema import DEFAULT_PILOT_CONFIG_PATH_V1_3

assert HARD_WEAK_LABEL_TRAINING_ALLOWED is False
assert SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED is False

cfg_v14 = None
cfg_v13 = None
if SOURCE_ROOT is not None:
    cfg_v14 = str(Path(SOURCE_ROOT) / "configs" / "second_reader_pilot_v1_4.yaml")
    cfg_v13 = str(Path(SOURCE_ROOT) / "configs" / "second_reader_pilot_v1_3.yaml")
WEIGHTS = os.environ.get("SECOND_READER_WEIGHTS")
OUT_DIR = Path("/kaggle/working/rsna_second_reader_constrained_canary")

config = load_pilot_config(cfg_v14) if cfg_v14 else {}
weights_dir = locate_weights_directory(config.get("second_reader") or config, WEIGHTS)
if weights_dir is not None:
    inspection = inspect_termination_from_snapshot(weights_dir)
    print("termination_inspection=" + json.dumps(inspection, sort_keys=True, default=str)[:8000])
else:
    print("termination_inspection=weights_not_attached")

provenance = schema_hash_provenance()
print("schema_hash_provenance=" + json.dumps({
    "schema_raw_bytes_sha256": provenance["schema_raw_bytes_sha256"],
    "schema_canonical_json_sha256": provenance["schema_canonical_json_sha256"],
    "generation_schema_raw_bytes_sha256": provenance["generation_schema_raw_bytes_sha256"],
    "generation_schema_canonical_json_sha256": provenance["generation_schema_canonical_json_sha256"],
    "same_parsed_object": provenance["same_parsed_object"],
    "raw_differs_from_canonical_due_to_serialization": provenance["raw_differs_from_canonical_due_to_serialization"],
    "generation_schema_is_validation_schema": provenance["generation_schema_is_validation_schema"],
}, sort_keys=True))
if not provenance.get("same_parsed_object"):
    raise SystemExit("SCHEMA_SEMANTIC_MISMATCH")

schema_audit = audit_schema_against_lmfe()
print("schema_compatibility=" + json.dumps({k: schema_audit[k] for k in schema_audit if k != "fingerprint"}, sort_keys=True)[:4000])
if not schema_audit.get("ok"):
    raise SystemExit("SCHEMA_COMPATIBILITY_FAIL")

tok_dir = None
if weights_dir is not None:
    tok_dir = weights_dir
tokenizer_audit = tokenizer_prefix_tree_audit(tok_dir)
print("tokenizer_compatibility=" + json.dumps({k: tokenizer_audit[k] for k in tokenizer_audit if k not in {"skeleton"}}, sort_keys=True, default=str)[:4000])
if tokenizer_audit.get("skipped"):
    print("tokenizer_compatibility=skipped")
elif not tokenizer_audit.get("ok"):
    raise SystemExit("TOKENIZER_COMPATIBILITY_FAIL")

try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    print("gpu_name=" + gpu_name)
except Exception as exc:
    print("gpu_info_unavailable=" + str(exc))


In [ ]:
reset_constraint_stats()
smoke_started = time.time()
smoke = run_constrained_canary(
    output_dir=OUT_DIR / "smoke",
    config_path=cfg_v14,
    weights_path=weights_dir,
    smoke=True,
    n_repeats=1,
    intervention=None,
)
smoke["inference_seconds"] = round(time.time() - smoke_started, 3)
diag = smoke.get("generation_diagnostics") or {}
print("constrained_smoke_status=" + str(smoke.get("status")))
print("constrained_smoke=" + json.dumps(
    {
        "ok": smoke.get("ok"),
        "status": smoke.get("status"),
        "n_examples": smoke.get("n_examples"),
        "n_schema_valid": smoke.get("n_schema_valid"),
        "n_schema_valid_all_repeats": smoke.get("n_schema_valid_all_repeats"),
        "malformed_output_rate": smoke.get("malformed_output_rate"),
        "n_gold_studies_processed": smoke.get("n_gold_studies_processed"),
        "n_report_only_studies_processed": smoke.get("n_report_only_studies_processed"),
        "n_targets_is_object": diag.get("n_targets_is_object"),
        "n_targets_not_object": diag.get("n_targets_not_object"),
        "n_enum_violations": diag.get("n_enum_violations"),
        "n_extra_properties": diag.get("n_extra_properties"),
        "n_reaching_max_new_tokens": diag.get("n_reaching_max_new_tokens"),
        "n_containing_balanced_json": diag.get("n_containing_balanced_json"),
        "n_forbidden_extra_text": diag.get("n_forbidden_extra_text"),
        "inference_backend": smoke.get("inference_backend"),
        "inference_seconds": smoke.get("inference_seconds"),
        "intervention_source": (smoke.get("intervention") or {}).get("source"),
    },
    sort_keys=True,
    default=str,
)[:8000])
try:
    import torch
    if torch.cuda.is_available():
        print("smoke_peak_gpu_memory_bytes=" + str(int(torch.cuda.max_memory_allocated())))
except Exception:
    pass
if not smoke.get("ok"):
    raise SystemExit("CONSTRAINED_SMOKE_FAIL")
if smoke.get("inference_backend") == "fixture":
    raise SystemExit("CONSTRAINED_SMOKE_FAIL: fixture generate_fn is not allowed on GPU")


In [ ]:
# Unconstrained v1.3 field capture, then two constrained v1.4 runs.
# Synthetic reports only. Completions and report text are not printed.
reset_constraint_stats()
unconstrained = run_synthetic_canary(
    output_dir=OUT_DIR / "unconstrained_v13_capture",
    config_path=cfg_v13 or str(DEFAULT_PILOT_CONFIG_PATH_V1_3),
    weights_path=weights_dir,
    n_repeats=1,
    include_clinical_fields=True,
    include_synthetic_truth=True,
)
unconstrained_rows = unconstrained.get("scored_rows") or []
print("unconstrained_capture_n_schema_valid=" + str(unconstrained.get("n_schema_valid")))
print("unconstrained_capture_n_gold=" + str(unconstrained.get("n_gold_studies_processed")))
print("unconstrained_example_ok=" + json.dumps(
    [{"example_id": row.get("example_id"), "ok": row.get("ok")} for row in unconstrained_rows],
    sort_keys=True,
))

reset_constraint_stats()
canary_started = time.time()
canary = run_constrained_canary(
    output_dir=OUT_DIR / "canary",
    config_path=cfg_v14,
    weights_path=weights_dir,
    n_repeats=2,
    unconstrained_rows=unconstrained_rows,
    intervention=None,
)
canary["inference_seconds"] = round(time.time() - canary_started, 3)
semantic = canary.get("semantic") or {}
intervention = canary.get("intervention") or aggregate_constraint_stats()
peak_gpu = None
try:
    import torch
    if torch.cuda.is_available():
        peak_gpu = int(torch.cuda.max_memory_allocated())
except Exception:
    peak_gpu = None
print("constrained_canary_status=" + str(canary.get("status")))
print("constrained_canary=" + json.dumps(
    {
        "ok": canary.get("ok"),
        "status": canary.get("status"),
        "n_examples": canary.get("n_examples"),
        "n_repeats": canary.get("n_repeats"),
        "n_completions": canary.get("n_completions"),
        "n_schema_valid": canary.get("n_schema_valid"),
        "n_schema_valid_all_repeats": canary.get("n_schema_valid_all_repeats"),
        "malformed_output_rate": canary.get("malformed_output_rate"),
        "repeatable": canary.get("repeatable"),
        "n_gold_studies_processed": canary.get("n_gold_studies_processed"),
        "n_report_only_studies_processed": canary.get("n_report_only_studies_processed"),
        "stop_gold_rerun": canary.get("stop_gold_rerun"),
        "gold_rerun_authorized": canary.get("gold_rerun_authorized"),
        "gold_rerun_executed": canary.get("gold_rerun_executed"),
        "inference_backend": canary.get("inference_backend"),
        "intervention": intervention,
        "semantic_status": semantic.get("status"),
        "synthetic_status_accuracy": (semantic.get("synthetic_status") or {}).get("overall_status_accuracy"),
        "field_agreement": semantic.get("field_agreement"),
        "n_outputs_clinical_fields_changed": semantic.get("n_outputs_clinical_fields_changed"),
        "n_target_level_clinical_decisions_changed": semantic.get("n_target_level_clinical_decisions_changed"),
        "code_never_coerces_unknown_to_negative": semantic.get("code_never_coerces_unknown_to_negative"),
        "inference_seconds": canary.get("inference_seconds"),
        "peak_gpu_memory_bytes": peak_gpu,
        "generation_diagnostics": canary.get("generation_diagnostics"),
    },
    sort_keys=True,
    default=str,
)[:12000])
print("gold_rerun_authorized=false gold_rerun_executed=false notebook_07_executed=false")
if canary.get("inference_backend") == "fixture":
    raise SystemExit("CONSTRAINED_CANARY_FORMAT_FAIL: fixture generate_fn is not allowed on GPU")
if not canary.get("ok"):
    raise SystemExit(str(canary.get("status") or "CANARY_FAIL"))
print("CONSTRAINED_CANARY_PASS")
